# Add comments to code 
## Uses Ollama, Openrouter and Gemini

In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


In [4]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



Google API Key exists and begins AI
OpenRouter API Key exists and begins sk-or-


In [5]:
# Urls

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)



In [21]:
models = [ "gemini-3.8-flash", "deepseek-coder-v2", "qwen/qwen3-coder-30b-a3b-instruct"]

clients = { "gemini-3.8-flash": gemini, "deepseek-coder-v2": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}


In [22]:

def create_comment_prompt(code):
    system_prompt = """
You are a code documentation assistant.

Your task is to add meaningful comments to the provided source code wherever comments would improve understanding.

Rules:
1. Return ONLY the modified source code.
2. Do NOT return Markdown code fences.
3. Do NOT add explanations, introductions, summaries, or any other text.
4. Preserve the original code's functionality exactly.
5. Do NOT modify, remove, or reorder existing code unless absolutely necessary to add a comment.
6. Preserve all existing comments unless they are incorrect or misleading.
7. Add comments only where they provide meaningful value.
8. Do not comment obvious code or every individual line.
9. Prefer comments that explain:
   - Why something is being done.
   - Non-obvious business logic.
   - Complex algorithms or calculations.
   - Important edge cases.
   - Performance considerations.
   - Workarounds or constraints.
   - Non-obvious dependencies between pieces of code.
10. Keep comments concise and technically accurate.
11. Place each comment as close as possible to the code it explains.
12. Use the idiomatic comment syntax of the programming language.
13. Never invent behavior, requirements, or business rules that cannot be inferred from the code.
14. Do not implement missing functionality.
15. The output must be valid source code for the specified language.

Your entire response must consist of the resulting source code and nothing else.
""".strip()

    user_prompt = f"""

Return the complete code with comments added.

CODE:
{code}
""".strip()
    return [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]




In [23]:
code = """
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None


class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, data):
        new_node = Node(data)

        if self.head is None:
            self.head = new_node
            return

        current = self.head

        while current.next:
            current = current.next

        current.next = new_node

    def prepend(self, data):
        new_node = Node(data)
        new_node.next = self.head
        self.head = new_node

    def delete(self, data):
        if self.head is None:
            return

        if self.head.data == data:
            self.head = self.head.next
            return

        current = self.head

        while current.next:
            if current.next.data == data:
                current.next = current.next.next
                return

            current = current.next

    def search(self, data):
        current = self.head

        while current:
            if current.data == data:
                return True

            current = current.next

        return False

    def reverse(self):
        previous = None
        current = self.head

        while current:
            next_node = current.next
            current.next = previous
            previous = current
            current = next_node

        self.head = previous

    def display(self):
        current = self.head
        values = []

        while current:
            values.append(str(current.data))
            current = current.next

        print(" -> ".join(values))


linked_list = LinkedList()

linked_list.append(10)
linked_list.append(20)
linked_list.append(30)
linked_list.prepend(5)

linked_list.display()

print(linked_list.search(20))
print(linked_list.search(50))

linked_list.delete(20)
linked_list.display()

linked_list.reverse()
linked_list.display()
"""

In [25]:
def addComment(model, pyCode):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=create_comment_prompt(pyCode), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    return reply

In [26]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Add comments to your Python code") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Original Code",
                value=code,
                language="python",
                lines=15
            )
        with gr.Column(scale=6):
            python_with_comments = gr.Code(
                label=f"Code with comments",
                value="",
                language="python",
                lines=15
            )

    with gr.Row(elem_classes=["controls"]):
        model = gr.Dropdown(models, value=models[0], show_label=False)
        runCommentAdd = gr.Button(f"Add comments", elem_classes=["add-comment-btn"])

    runCommentAdd.click(fn=addComment, inputs=[model, python], outputs=[python_with_comments])    


ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
